Gold — RAG projection

In [0]:
from pyspark.sql import functions as F

LABEL_SECTIONS = ["warnings", "adverse_reactions", "contraindications",
                  "drug_interactions", "dosage_and_administration",
                  "warnings_and_cautions", "boxed_warning"]

labels = spark.table("fda_rag.silver.drug_labels")
events = spark.table("fda_rag.silver.adverse_events")

label_chunks_dfs = []
for section in LABEL_SECTIONS:
    df = (labels
        .filter(F.col(section).isNotNull() & (F.length(section) > 50))
        .select(
            F.col("drug_brand"),
            F.col("drug_generic"),
            F.lit(section).alias("section"),
            F.lit("label").alias("source_type"),
            F.substring(F.col(section), 1, 4000).alias("text")
        ))
    label_chunks_dfs.append(df)

label_chunks = label_chunks_dfs[0]
for d in label_chunks_dfs[1:]:
    label_chunks = label_chunks.unionByName(d)

event_chunks = (events
    .withColumn("text", F.concat(
        F.lit("Patient age "),
        F.coalesce(F.col("patient_age").cast("string"), F.lit("unknown")),
        F.lit(", sex "), F.col("patient_sex"),
        F.lit(". Serious: "), F.coalesce(F.col("serious"), F.lit("unknown")),
        F.lit(". Reported reactions: "),
        F.array_join(F.col("reactions"), ", ")
    ))
    .select(
        F.col("drug_name").alias("drug_brand"),
        F.col("drug_name").alias("drug_generic"),
        F.lit("adverse_event_report").alias("section"),
        F.lit("event").alias("source_type"),
        F.col("text")
    )
    .filter(F.length("text") > 50))

chunks = (label_chunks.unionByName(event_chunks)
    .withColumn("chunk_id",
        F.sha2(F.concat_ws("|", "drug_generic", "section", "text"), 256))
    .dropDuplicates(["chunk_id"]))

(chunks.write.mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .saveAsTable("fda_rag.gold.fda_chunks"))

print(f"Gold chunks: {spark.table('fda_rag.gold.fda_chunks').count()}")